# Chapter 9 Simulations — Analysis Stage: Grid Search, Ground-Truth Evaluation & Judge Calibration

This notebook runs three evaluations against the Stage 4 analysis pipeline defined in `my_agent/agent.py`:

1. **Grid Search** — three models × three temperatures × ten analysis scenarios × five Monte Carlo simulations = 450 total runs. Ranked by F1 (judge-compliance metric). Writes the winning configuration to `best_config.json`.
2. **Ground-Truth Evaluation** — five expert-curated scenarios from `ground_truth.csv`. Pipeline output is compared against analyst-expected SAT selections via F1 score.
3. **Judge Calibration** — fourteen test cases with known-correct verdicts test whether the SAT Judge correctly identifies hallucinated techniques, missing SATs, and technique-scenario mismatches.

**Thresholds:**
- F1 ≥ 70%
- Judge accuracy ≥ 80%

> **Note on recorded results.** This notebook ships with its outputs cleared. The results reproduced in the chapter come from the recorded run saved in `ground_truth.csv` (ground-truth evaluation: average F1 0.78, PASS) and `my_agent/judge_eval_results.json` (judge calibration: 12/14, 86%), produced with the `gemini-3.1-pro-preview` generator recorded in `my_agent/best_config.json`. Re-running the evaluation cells regenerates and overwrites those files, and the numbers may vary slightly from run to run.

## 1. Setup

Load environment variables and import the pipeline. `run_pipeline()` is imported directly from `my_agent/agent.py` — it constructs fresh agent instances per call and manages its own session state.

In [ ]:
import asyncio
import json
import os
import sys
import csv
import time
from pathlib import Path
from collections import defaultdict

import pandas as pd
from IPython.display import HTML, display

from dotenv import load_dotenv
load_dotenv()

sys.path.insert(0, str(Path("my_agent").resolve()))

from agent import run_pipeline, DEFAULT_MODEL, DEFAULT_TEMP
from domain_data import SAT_CATALOG, SAT_NAMES, TEMPLATE_VERSION, CONFIG_SEARCH_INPUTS
from judge_eval import JUDGE_TEST_CASES, evaluate_judge

JUDGE_ACCURACY_THRESHOLD = 80.0
GT_F1_THRESHOLD = 0.70

print(f"Default model  : {DEFAULT_MODEL} @ temp={DEFAULT_TEMP}")
print(f"Template ver   : {TEMPLATE_VERSION}")
print(f"SAT catalog    : {len(SAT_CATALOG)} techniques")
print(f"Judge cases    : {len(JUDGE_TEST_CASES)}")

## 2. Pipeline Helper

Wraps a single end-to-end run of `run_pipeline()` from `agent.py`. The agent constructs fresh instances per call. Returns `(session_id, iterations, final_verdict)` with verbose per-iteration output.

In [ ]:
async def run_scenario(
    corroborated_brief: str,
    threat_context: str = "",
    max_iterations: int = 3,
    verbose: bool = True,
) -> tuple[str, list[dict], dict]:
    """Run the pipeline on one scenario and return (session_id, iterations, final_verdict)."""
    if verbose:
        print(f"Model: {DEFAULT_MODEL} @ temp={DEFAULT_TEMP}  max_iter={max_iterations}")
        print()

    session_id, iterations = await run_pipeline(
        corroborated_brief=corroborated_brief,
        max_iterations=max_iterations,
        threat_context=threat_context,
    )

    if verbose:
        sep = "─" * 60
        for it in iterations:
            n = it["iteration"]
            verdict = it["verdict"]
            n_v = len(verdict.get("confirmed_valid", []))
            n_u = len(verdict.get("unverified", []))
            n_m = len(verdict.get("missing_critical", []))
            label = verdict.get("verdict", "UNKNOWN")
            print(sep)
            print(f"Iteration {n} — {label}")
            print(f"  valid={n_v}  unverified={n_u}  missing={n_m}")
            print(f"  {verdict.get('summary', '')[:120]}")
            print()

        final = iterations[-1]["verdict"]
        print("=" * 60)
        print(f"Final verdict : {final.get('verdict', 'UNKNOWN')} in {len(iterations)} iteration(s)")

    return session_id, iterations, iterations[-1]["verdict"]

## 3. Single-Scenario Demo

Runs the AiTM Session Hijacking scenario — the canonical threat for ApexCode's partner access environment. Demonstrates the full self-refining loop: Generator Agent selects and applies SATs, SAT Judge validates, Verification Agent corrects.

In [ ]:
DEMO_BRIEF = (
    "HIGH CONFIDENCE: Partner developer at DevPartner Inc. account used to clone "
    "3 sensitive repos (phoenix-core, phoenix-api, phoenix-auth) from IP 185.220.101.x "
    "(known UNC-XXXX egress point). Developer's credentials appeared in dark web dump 48h "
    "prior. Impossible travel detected: US login, then RU-based clone within 20min. "
    "CrowdStrike process alert on partner workstation shows Evilginx2 proxy artifact."
)

demo_session_id, demo_iterations, demo_verdict = await run_scenario(
    corroborated_brief=DEMO_BRIEF,
    verbose=True,
)

In [ ]:
# Spot-check: run the pipeline once to verify it works before committing to a full grid search.

test_brief = (
    "HIGH CONFIDENCE: Partner developer account used to clone 3 sensitive repos. "
    "Evilginx2 proxy artifact detected. Impossible travel from US to RU in 20 min."
)

session_id, iterations = await run_pipeline(test_brief, max_iterations=2)

for it in iterations:
    verdict = it["verdict"]
    print(f"Iteration {it['iteration']}: {verdict.get('verdict')} "
          f"(valid={len(verdict.get('confirmed_valid',[]))} "
          f"unverified={len(verdict.get('unverified',[]))} "
          f"missing={len(verdict.get('missing_critical',[]))})")

## 4. Grid Search

The sweep covers three models \u00d7 three temperatures \u00d7 ten analysis scenarios \u00d7 five Monte Carlo simulations = 450 total runs, capped at 5 concurrent pipelines via `asyncio.Semaphore`. Each scenario exercises a distinct threat vector.

Ranked by average F1; ties broken by average iteration count. The winning configuration is written to `best_config.json`.

In [ ]:
MODELS = ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.1-pro-preview"]
TEMPERATURES = [0.0, 0.2, 0.5]
MONTE_CARLO_RUNS = 5
CONCURRENCY_LIMIT = 5
GS_RESULTS_CSV = Path("config_search_results.csv")

# 10 scenarios from domain_data.py
GS_SCENARIOS = CONFIG_SEARCH_INPUTS

print(f"Grid: {len(MODELS)} models \u00d7 {len(TEMPERATURES)} temps "
      f"\u00d7 {len(GS_SCENARIOS)} scenarios \u00d7 {MONTE_CARLO_RUNS} sims "
      f"= {len(MODELS)*len(TEMPERATURES)*len(GS_SCENARIOS)*MONTE_CARLO_RUNS} total runs")

In [ ]:
async def _run_single(model, temperature, scenario, run_idx):
    start = time.monotonic()
    _, iterations = await run_pipeline(
        corroborated_brief=scenario["corroborated_brief"],
        max_iterations=3,
        analysis_model=model,
        analysis_temp=temperature,
    )
    latency = time.monotonic() - start
    final = iterations[-1]["verdict"]
    valid   = len(final.get("confirmed_valid", []))
    unver   = len(final.get("unverified", []))
    missing = len(final.get("missing_critical", []))
    total   = valid + unver
    precision = valid / total if total > 0 else 0.0
    coverage  = valid / (valid + missing) if (valid + missing) > 0 else 0.0
    f1 = (2 * precision * coverage) / (precision + coverage) if (precision + coverage) > 0 else 0.0
    return {
        "model": model, "temperature": temperature,
        "scenario": scenario.get("label", scenario.get("name", "")), "run": run_idx,
        "precision": precision, "coverage": coverage, "f1": f1,
        "n_iters": len(iterations), "latency_s": round(latency, 2),
        "verdict": final.get("verdict", "UNKNOWN"),
    }

async def run_grid():
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    all_results = []

    async def bounded(model, temp, scenario, run_idx):
        async with semaphore:
            return await _run_single(model, temp, scenario, run_idx)

    tasks = [
        bounded(model, temp, scenario, run_idx)
        for model in MODELS
        for temp in TEMPERATURES
        for scenario in GS_SCENARIOS
        for run_idx in range(1, MONTE_CARLO_RUNS + 1)
    ]

    print(f"Starting {len(tasks)} executions (concurrency={CONCURRENCY_LIMIT})...")
    for coro in asyncio.as_completed(tasks):
        result = await coro
        all_results.append(result)
        if len(all_results) % 25 == 0:
            print(f"  {len(all_results)}/{len(tasks)} complete")

    agg = defaultdict(list)
    for r in all_results:
        agg[(r["model"], r["temperature"])].append(r)

    summary = []
    for (model, temp), runs in sorted(agg.items()):
        summary.append({
            "model": model, "temperature": temp,
            "avg_f1": round(sum(r["f1"] for r in runs) / len(runs), 3),
            "avg_precision": round(sum(r["precision"] for r in runs) / len(runs), 3),
            "avg_coverage": round(sum(r["coverage"] for r in runs) / len(runs), 3),
            "avg_iters": round(sum(r["n_iters"] for r in runs) / len(runs), 2),
            "avg_latency_s": round(sum(r["latency_s"] for r in runs) / len(runs), 2),
            "pass_rate": round(sum(1 for r in runs if r["verdict"] == "PASS") / len(runs), 3),
        })
    summary.sort(key=lambda x: x["avg_f1"], reverse=True)

    fieldnames = ["model", "temperature", "avg_f1", "avg_precision", "avg_coverage",
                  "avg_iters", "pass_rate", "avg_latency_s"]
    with GS_RESULTS_CSV.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary)

    best = summary[0]
    best_path = Path("my_agent/best_config.json")
    best_path.write_text(json.dumps({
        "model": best["model"],
        "temperature": best["temperature"],
        "template_version": TEMPLATE_VERSION,
    }, indent=2))

    print(f"\nResults written to {GS_RESULTS_CSV}")
    print(f"Best config: {best['model']} @ temp={best['temperature']} (avg F1={best['avg_f1']:.3f})")
    return summary

gs_summary = await run_grid()

In [ ]:
# Reload config_search_results.csv and render a ranked Styler table.

if not GS_RESULTS_CSV.exists() or GS_RESULTS_CSV.stat().st_size == 0 or pd.read_csv(GS_RESULTS_CSV).empty:
    print("No grid search results yet \u2014 run the grid search cell above first.")
else:
    _df_gs = pd.read_csv(GS_RESULTS_CSV)
    _df_gs_disp = _df_gs.rename(columns={
        "model":         "Model",
        "temperature":   "Temp",
        "avg_f1":        "avg F1",
        "avg_iters":     "Iters",
        "pass_rate":     "Pass %",
        "avg_latency_s": "Latency (s)",
        "n_runs":        "Runs",
    }).reset_index(drop=True)
    _df_gs_disp.index = _df_gs_disp.index + 1  # 1-based rank.

    def _color_f1(v):
        if v >= 0.90: return "background-color: #dcfce7; color: #166534; font-weight: 600"
        if v >= 0.70: return "background-color: #fef9c3; color: #92400e"
        return "background-color: #fee2e2; color: #991b1b"

    def _hl_best(row):
        return ["background-color: #eff6ff"] * len(row) if row.name == 1 else [""] * len(row)

    _gs_tbl_styles = [
        {"selector": "caption", "props": [("font-size", "0.9rem"), ("font-weight", "600"),
                                           ("padding", "0.4rem 0"), ("text-align", "left")]},
        {"selector": "th",      "props": [("font-size", "0.78rem"), ("text-transform", "uppercase"),
                                           ("color", "#6b7280"), ("padding", "0.35rem 0.55rem")]},
        {"selector": "td",      "props": [("padding", "0.35rem 0.55rem"), ("font-size", "0.83rem")]},
    ]

    display(
        _df_gs_disp.style
        .apply(_hl_best, axis=1)
        .map(_color_f1, subset=["avg F1"])
        .format({"avg F1": "{:.3f}", "Pass %": "{:.0%}",
                 "Iters": "{:.2f}", "Latency (s)": "{:.1f}s",
                 "Temp": "{:.1f}"})
        .set_caption(f"Configuration Grid Search \u2014 {len(_df_gs)} configurations, ranked by avg F1")
        .set_table_styles(_gs_tbl_styles)
    )


In [ ]:
# Model x Temperature F1 pivot table.

if not GS_RESULTS_CSV.exists() or GS_RESULTS_CSV.stat().st_size == 0 or pd.read_csv(GS_RESULTS_CSV).empty:
    print("No grid search results yet.")
else:
    _df_pivot = _df_gs.pivot_table(
        index="model", columns="temperature", values="avg_f1", aggfunc="mean"
    ).round(3)
    _df_pivot.index.name = "Model"
    _df_pivot.columns.name = "Temperature"

    def _color_cell(v):
        if pd.isna(v): return ""
        if v >= 0.90: return "background-color: #dcfce7; color: #166534; font-weight: 600"
        if v >= 0.70: return "background-color: #fef9c3; color: #92400e"
        return "background-color: #fee2e2; color: #991b1b"

    display(
        _df_pivot.style
        .map(_color_cell)
        .format("{:.3f}")
        .set_caption("avg F1 by Model \u00d7 Temperature")
    )


## 5. Ground-Truth Evaluation

Runs the pipeline against five expert-curated scenarios from `ground_truth.csv` and compares the SAT selections to analyst expectations via F1 score.

- **F1 score** = harmonic mean of precision (pipeline SATs that are correct) and recall (expected SATs the pipeline found). Threshold: ≥ 70%.

In [ ]:
GT_CSV = Path("ground_truth.csv")
GROUND_TRUTH = []
with GT_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        GROUND_TRUTH.append({
            "name": row["Scenario"],
            "corroborated_brief": row["corroborated_brief"],
            "expected_sats": json.loads(row["expected_sats"]),
            "expected_key_finding": row["expected_key_finding"],
        })

def compute_f1(set_a: set, set_b: set) -> float:
    if not set_a and not set_b:
        return 1.0
    intersection = len(set_a & set_b)
    precision = intersection / len(set_a) if set_a else 0.0
    recall    = intersection / len(set_b) if set_b else 0.0
    return (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

def extract_sats_from_output(output: str) -> set:
    found = set()
    output_upper = output.upper()
    for name in SAT_NAMES:
        if name.upper() in output_upper:
            found.add(name)
    return found

print(f"Loaded {len(GROUND_TRUTH)} ground truth scenarios from {GT_CSV}")
for i, gt in enumerate(GROUND_TRUTH, 1):
    print(f"  {i}. {gt['name'][:55]}  ({len(gt['expected_sats'])} expected SATs)")

In [ ]:
import csv as _csv_gt

sep = "─" * 60

# ── Resume logic ─────────────────────────────────────────────────────────────
# 1. Keep any results already in memory from a partial run this session.
# 2. If gt_results is empty or undefined, reload completed rows from the CSV
#    so a kernel restart doesn't force re-running finished scenarios.
try:
    _existing = {r["name"] for r in gt_results}
except NameError:
    gt_results = []
    _existing = set()

if not _existing:
    with GT_CSV.open(newline="", encoding="utf-8") as _f:
        for _row in _csv_gt.DictReader(_f):
            if _row.get("F1") and _row["F1"] not in ("0.0", ""):
                _exp = json.loads(_row["expected_sats"])
                gt_results.append({
                    "name": _row["Scenario"],
                    "expected_sats": _exp,
                    "pipeline_sats": [],
                    "f1": float(_row["F1"]),
                    "matched": [],
                    "missed": [s for s in _row.get("Missed", "").split(",") if s],
                    "extra":  [s for s in _row.get("Extra",  "").split(",") if s],
                    "n_iters": int(_row["Iters"]) if _row.get("Iters") else 0,
                    "verdict": _row.get("Verdict", "UNKNOWN"),
                    "n_pipeline": int(_row["Pipeline_Findings"]) if _row.get("Pipeline_Findings") else 0,
                    "n_matched":  int(_row["Overlap"])           if _row.get("Overlap")           else 0,
                })
    _existing = {r["name"] for r in gt_results}

if _existing:
    print(f"Resuming — {len(_existing)} scenario(s) already complete: {sorted(_existing)}")
# ─────────────────────────────────────────────────────────────────────────────

for i, gt in enumerate(GROUND_TRUTH, 1):
    if gt["name"] in _existing:
        print(f"\nScenario {i}/{len(GROUND_TRUTH)}: {gt['name'][:52]} — skipped (already done)")
        continue

    print("\n" + sep)
    print(f"Scenario {i}/{len(GROUND_TRUTH)}: {gt['name']}")

    _, iterations = await run_pipeline(
        corroborated_brief=gt["corroborated_brief"],
        max_iterations=3,
    )

    verified = iterations[-1].get("verified_analysis", "") if iterations else ""
    pipeline_sats = extract_sats_from_output(verified)
    expected_sats = set(gt["expected_sats"])
    f1_score = compute_f1(pipeline_sats, expected_sats)

    result = {
        "name": gt["name"],
        "expected_sats": sorted(expected_sats),
        "pipeline_sats": sorted(pipeline_sats),
        "f1": round(f1_score, 3),
        "matched": sorted(pipeline_sats & expected_sats),
        "missed": sorted(expected_sats - pipeline_sats),
        "extra": sorted(pipeline_sats - expected_sats),
        "n_iters": len(iterations),
        "verdict": iterations[-1].get("verdict", {}).get("verdict", "UNKNOWN"),
    }
    gt_results.append(result)
    _existing.add(gt["name"])

    print(f"  F1: {f1_score:.1%}  Expected: {sorted(expected_sats)}  Got: {sorted(pipeline_sats)}")
    if expected_sats - pipeline_sats:
        print(f"  Missed: {sorted(expected_sats - pipeline_sats)}")
    if pipeline_sats - expected_sats:
        print(f"  Extra: {sorted(pipeline_sats - expected_sats)}")

    # ── Checkpoint: write this result to CSV immediately ─────────────────────
    _csv_rows = []
    with GT_CSV.open(newline="", encoding="utf-8") as _f:
        _rdr = _csv_gt.DictReader(_f)
        _fieldnames = _rdr.fieldnames
        _csv_rows = list(_rdr)
    for _row in _csv_rows:
        if _row["Scenario"] == result["name"]:
            _row["Pipeline_Findings"] = len(result["pipeline_sats"])
            _row["GT_Findings"]       = len(result["expected_sats"])
            _row["Overlap"]           = len(result["matched"])
            _row["Missed"]            = ",".join(result["missed"])
            _row["Extra"]             = ",".join(result["extra"])
            _row["F1"]                = result["f1"]
            _row["Verdict"]           = result["verdict"]
            _row["Iters"]             = result["n_iters"]
    with GT_CSV.open("w", newline="", encoding="utf-8") as _f:
        _wtr = _csv_gt.DictWriter(_f, fieldnames=_fieldnames, quoting=_csv_gt.QUOTE_ALL)
        _wtr.writeheader()
        _wtr.writerows(_csv_rows)
    # ─────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print(f"GT EVALUATION COMPLETE  ({len(gt_results)}/{len(GROUND_TRUTH)} scenarios)")


In [ ]:
avg_f1 = sum(r["f1"] for r in gt_results) / len(gt_results)

def _style_gt(rows):
    df = pd.DataFrame([{
        "Scenario": r["name"][:52],
        "Expected": len(r["expected_sats"]),
        "Pipeline": r.get("n_pipeline", len(r["pipeline_sats"])),
        "Matched":  r.get("n_matched",  len(r["matched"])),
        "Missed":   ", ".join(r["missed"]) or "\u2014",
        "Extra":    ", ".join(r["extra"])  or "\u2014",
        "F1":       r["f1"],
        "Verdict":  r["verdict"],
        "Iters":    r["n_iters"],
    } for r in rows])

    def _color_f1(v):
        if v >= 0.90: return "background-color: #dcfce7; color: #166534; font-weight: 600"
        if v >= 0.70: return "background-color: #fef9c3; color: #92400e"
        return "background-color: #fee2e2; color: #991b1b"

    def _verdict_color(v):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
        }.get(str(v), "color: #64748b")

    status = "PASS" if avg_f1 >= GT_F1_THRESHOLD else "FAIL"

    _gt_tbl_styles = [
        {"selector": "caption", "props": [("font-size", "0.9rem"), ("font-weight", "600"),
                                           ("padding", "0.4rem 0"), ("text-align", "left")]},
        {"selector": "th",      "props": [("font-size", "0.78rem"), ("text-transform", "uppercase"),
                                           ("color", "#6b7280"), ("padding", "0.35rem 0.55rem")]},
        {"selector": "td",      "props": [("padding", "0.35rem 0.55rem"), ("font-size", "0.83rem")]},
    ]

    return (
        df.style
        .map(_color_f1,      subset=["F1"])
        .map(_verdict_color, subset=["Verdict"])
        .format({"F1": "{:.3f}"})
        .set_caption(
            f"Ground-Truth Evaluation \u2014 {len(rows)} scenarios  |  "
            f"F1 \u2265 {GT_F1_THRESHOLD:.0%}"
        )
        .set_table_styles(_gt_tbl_styles)
        .hide(axis="index")
    )

display(_style_gt(gt_results))

print(f"\navg F1: {avg_f1:.1%}  (threshold {GT_F1_THRESHOLD:.0%})"
      f"  {'PASS' if avg_f1 >= GT_F1_THRESHOLD else 'FAIL'}")

# Write results back to CSV
import csv as _csv
existing_rows = []
with GT_CSV.open(newline="", encoding="utf-8") as f:
    reader = _csv.DictReader(f)
    fieldnames = reader.fieldnames
    for row in reader:
        existing_rows.append(row)

result_by_name = {r["name"]: r for r in gt_results}
for row in existing_rows:
    r = result_by_name.get(row["Scenario"])
    if r:
        # Resumed entries carry the CSV's true counts in n_pipeline/n_matched
        # (their list fields are empty); fresh runs fall through to len().
        row["Pipeline_Findings"] = r.get("n_pipeline", len(r["pipeline_sats"]))
        row["GT_Findings"]       = len(r["expected_sats"])
        row["Overlap"]           = r.get("n_matched",  len(r["matched"]))
        row["Missed"]            = ",".join(r["missed"])
        row["Extra"]             = ",".join(r["extra"])
        row["F1"]                = r["f1"]
        row["Verdict"]           = r["verdict"]
        row["Iters"]             = r["n_iters"]

with GT_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = _csv.DictWriter(f, fieldnames=fieldnames, quoting=_csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(existing_rows)

print(f"Wrote results to: {GT_CSV}")


In [ ]:
saved_rows = []
with GT_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row.get("F1") and row["F1"] != "0.0":
            saved_rows.append(row)

if not saved_rows:
    print("No results in ground_truth.csv yet — run the GT evaluation first.")
else:
    print(f"{'Scenario':<55s}  {'F1':>6}  {'Verdict'}")
    print("-" * 75)
    for row in saved_rows:
        f1 = float(row["F1"]) if row["F1"] else 0.0
        print(f"{row['Scenario']:<55s}  {f1:>6.0%}  {row['Verdict']}")

## 6. Judge Calibration

Fourteen test cases with known-correct verdicts test whether the SAT Judge correctly identifies hallucinated techniques, missing SATs, taxonomy violations, and technique-scenario mismatches.

**Threshold:** \u2265 80% of checks must pass.

In [ ]:
print("=" * 70)
print("JUDGE CALIBRATION \u2014 Analysis Stage")
print(f"Test cases : {len(JUDGE_TEST_CASES)}")
print(f"Threshold  : {JUDGE_ACCURACY_THRESHOLD:.0f}%")
print("=" * 70)

judge_summary = await evaluate_judge()

eval_rows = []
for r in judge_summary.get("results", []):
    eval_rows.append({
        "name": r.get("id", "?"),
        "description": r.get("description", "")[:80],
        "expected_verdict": r.get("expected_verdict", "?"),
        "actual_verdict": r.get("actual_verdict", "?"),
        "passed": r.get("verdict_match", False),
        "summary": r.get("description", "")[:100],
    })

n_pass = sum(1 for r in eval_rows if r["passed"])
n_total = len(eval_rows)
accuracy = n_pass / n_total * 100 if n_total else 0
status = "PASS" if accuracy >= JUDGE_ACCURACY_THRESHOLD else "FAIL"
print(f"\nAccuracy: {accuracy:.0f}%  (threshold {JUDGE_ACCURACY_THRESHOLD:.0f}%) \u2014 {status}")

In [ ]:
def _style_judge_eval(rows: list) -> "pd.io.formats.style.Styler":
    df = pd.DataFrame([{
        "Test Case": r["name"],
        "Expected":  r["expected_verdict"],
        "Actual":    r["actual_verdict"],
        "Result":    "OK" if r["passed"] else "FAIL",
        "Summary":   r.get("summary", r.get("description", ""))[:100],
    } for r in rows])

    n_pass  = sum(1 for r in rows if r["passed"])
    n_total = len(rows)
    acc     = (n_pass / n_total * 100.0) if n_total else 0.0
    cap_status = "PASS" if acc >= JUDGE_ACCURACY_THRESHOLD else "FAIL"

    def _row_color(row):
        return (["background-color: #dcfce7"] * len(row) if row["Result"] == "OK"
                else ["background-color: #fee2e2"] * len(row))

    def _verdict_color(val):
        return {
            "PASS":    "color: #15803d; font-weight: 600",
            "PARTIAL": "color: #ca8a04; font-weight: 600",
            "FAIL":    "color: #dc2626; font-weight: 600",
            "ERROR":   "color: #7c3aed; font-weight: 600",
        }.get(str(val), "color: #64748b")

    return (
        df.style
        .apply(_row_color, axis=1)
        .map(_verdict_color, subset=["Expected", "Actual"])
        .set_caption(
            f"Judge Calibration \u2014 {n_pass}/{n_total} cases fully correct "
            f"({acc:.0f}%) \u2014 threshold {JUDGE_ACCURACY_THRESHOLD:.0f}% \u2014 {cap_status}"
        )
        .set_table_styles([
            {"selector": "caption",
             "props": [("font-size", "0.9rem"), ("font-weight", "700"),
                       ("color", "#1e293b"), ("padding-bottom", "10px"),
                       ("text-align", "left")]},
            {"selector": "th",
             "props": [("background-color", "#f1f5f9"), ("color", "#475569"),
                       ("font-size", "0.82rem"), ("padding", "8px 12px"),
                       ("border-bottom", "2px solid #e2e8f0")]},
            {"selector": "td",
             "props": [("font-size", "0.83rem"), ("padding", "7px 12px"),
                       ("border-bottom", "1px solid #f1f5f9"),
                       ("max-width", "320px"), ("word-wrap", "break-word")]},
        ])
        .hide(axis="index")
    )

if not eval_rows:
    print("No judge eval rows yet \u2014 run the judge calibration cell above first.")
else:
    display(_style_judge_eval(eval_rows))


## 7. Discussion

The three evaluations form a layered validation hierarchy, each catching failure modes the others cannot.

**Grid search** establishes whether the self-refining loop converges reliably across diverse analysis scenarios and model configurations. A ceiling effect \u2014 F1=100% across all configurations \u2014 indicates the SAT selection rules fully constrain the output space. When that ceiling is present, the winning configuration is the cheapest one with the lowest average iteration count.

**Ground-truth evaluation** breaks the judge's monopoly on correctness. A lenient judge can emit PASS verdicts on analysis outputs that select the wrong SATs for a given scenario. The F1 check surfaces this: if the pipeline's SAT selections don't match what a human analyst would choose, the analysis is wrong regardless of the judge's opinion.

**Judge calibration** validates the judge's own reliability. If the judge fails to catch hallucinated techniques (non-catalog SATs), misses mandatory SATs for a scenario, or accepts taxonomy violations, the F1 scores from the grid search are meaningless.

**SAT catalog as closed world**: The 14-technique catalog in `domain_data.py` is intentionally constrained. The pipeline cannot select techniques outside this catalog \u2014 any non-catalog technique in the output is a hallucination, caught by the judge. This closed-world constraint is what makes the evaluation tractable: correctness is binary and verifiable.